# Pipeline — Step-by-Step Test

This notebook walks through the pipeline stages one at a time:

| Stage | Function | What it does |
|-------|----------|-------------|
| Preprocess | `preprocess_page` | Binarise, deskew, mask borders & binding, crop |
| Segment | `segment_kraken` | Kraken text-line detection + figure extraction |
| Postprocess | `postprocess` | Placeholder — layout / line extraction to come |

Run the cells in order. Change `IMAGE_TAG` to try different exemplars.

## Setup

In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Make sure the project root is on the Python path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

EXEMPLARS_DIR = PROJECT_ROOT / "data" / "exemplars"

def pick(tag: str) -> Path:
    matches = sorted(EXEMPLARS_DIR.glob(f"*{tag}*.jpg"))
    if not matches:
        available = [p.stem.split("_", 1)[-1] for p in sorted(EXEMPLARS_DIR.glob("*.jpg"))]
        raise FileNotFoundError(
            f"No exemplar matching '{tag}'.\nAvailable tags:\n  " + "\n  ".join(available)
        )
    return matches[0]

all_images = sorted(EXEMPLARS_DIR.glob("*.jpg"))
print(f"Found {len(all_images)} exemplar images:")
for p in all_images:
    tag = "_".join(p.stem.split("_")[1:])
    print(f"  {tag}")


---
## Preprocess

Applies Sauvola binarisation, deskewing (±5°), masks page borders and binding shadow, and crops all outputs to the detected content area.

In [ ]:
from pipeline.stages import preprocess_page
from pipeline.plots  import plot_preprocess

IMAGE_TAG = "double_packed_center"   # ← change this to try other exemplars

img_path = pick(IMAGE_TAG)
print(f"Image: {img_path.name}")

img_path = "/Users/luissalamanca/Dropbox/My_stuff/05_SDSCresearch/10_SideProjects/00_MedievalCambridge/line_counting/data/all_images/MS-GG-00001-00001-000-00101.jpg"
pre = preprocess_page(img_path)

print("\n── Preprocess features ──")
print(f"  Image size (h × w)  : {pre.image_h} × {pre.image_w}")
print(f"  Deskew angle (°)    : {pre.deskew_angle:.2f}")
print(f"  Binding side        : {pre.binding_side}")
print(f"  Margin width (px)   : {pre.margin_width}")
print(f"  Top margin (px)     : {pre.top_margin}")
print(f"  Ruler height (px)   : {pre.ruler_height}")
print(f"  Bottom margin (px)  : {pre.bottom_margin}")
print(f"  Binding width (px)  : {pre.binding_width}")
print(f"  Crop rect (x,y,w,h) : {pre.crop_rect}")


In [ ]:
fig = plot_preprocess(pre)
fig.suptitle(f"Preprocess — {img_path.stem}", fontsize=13, fontweight="bold")
plt.show()


---
## Segment (Kraken)

Runs Kraken's neural baseline segmenter on the colour image to detect text-line polygons.
Also extracts figure / illustration regions as ink pixels that lie outside all text-line polygons.

Device: set `device="mps"`, `"cuda"`, or `"cpu"`.  
Model: `None` for Kraken's built-in default, or pass a path to a `.mlmodel` file.

In [ ]:
from pipeline.stages import segment_kraken
from pipeline.plots  import plot_segment_kraken

MODEL_PATH = "/Users/luissalamanca/Library/Application Support/htrmopo/97665cf3-f83d-5594-8855-f28d3af9df7a/blla.mlmodel"
DEVICE = "mps"

seg = segment_kraken(
    pre,
    model_path=MODEL_PATH,
    device=DEVICE,
    dilation_px=2,
)


In [ ]:
print("── Segment (Kraken) features ──")
print(f"  Text lines detected  : {seg.n_lines}")
print(f"  Text polygon coverage: {seg.text_coverage:.3f}")
print(f"  Foreground px in     : {seg.text_px_input:,}")
print(f"  Foreground px kept   : {seg.text_px_kept:,}")
if seg.text_px_input > 0:
    retained = seg.text_px_kept / seg.text_px_input * 100
    print(f"  Foreground retained  : {retained:.1f} %")
print(f"\n  Figure components    : {seg.n_figures}")
print(f"  Figure ink coverage  : {seg.figure_coverage:.4f}")
if seg.figure_bboxes:
    areas = [w * h for _, _, w, h in seg.figure_bboxes]
    print(f"  Largest figure (px²) : {max(areas):,}")


In [ ]:
fig = plot_segment_kraken(seg, pre=pre)
fig.suptitle(
    f"Segment (Kraken) — {img_path.stem}  |  "
    f"{seg.n_lines} lines  |  {seg.n_figures} figures",
    fontsize=13, fontweight="bold",
)
plt.show()


---
## Postprocess *(placeholder)*

This stage is currently empty and will be populated with layout detection, per-column line extraction, and bounding boxes in future iterations.

In [ ]:
from pipeline.stages import postprocess

post = postprocess(pre, seg)
print("postprocess() returned PostprocessResult — placeholder, no features yet.")
print(f"  type: {type(post).__name__}")


In [ ]:
# Placeholder visualisation — will be replaced once postprocess() is populated.
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.text(
    0.5, 0.5,
    "postprocess() output\n(to be implemented)",
    ha="center", va="center", fontsize=14, color="gray",
    transform=ax.transAxes,
)
ax.set_title("Postprocess — placeholder", fontsize=12)
ax.axis("off")
fig.tight_layout()
plt.show()


---
## Batch run — selection of exemplars

Run preprocess + segment on a representative set of images and display a feature table.

In [ ]:
import pandas as pd
from pipeline.stages import preprocess_page, segment_kraken

SAMPLE_TAGS = [
    "double_simple",
    "double_filigranes",
    "double_big_stain",
    "double_figures",
    "single_simple",
    "single_many_elements",
    "single_top_double_bottom",
    "single_separation_lines",
]

rows = []
for tag in SAMPLE_TAGS:
    try:
        path = pick(tag)
    except FileNotFoundError:
        continue
    print(f"Processing {path.name} ...", end=" ", flush=True)
    _pre = preprocess_page(path)
    _seg = segment_kraken(_pre, model_path=MODEL_PATH, device=DEVICE, dilation_px=2)
    rows.append({
        "image":           path.stem,
        "img_h":           _pre.image_h,
        "img_w":           _pre.image_w,
        "deskew_angle":    round(_pre.deskew_angle, 2),
        "binding_side":    _pre.binding_side,
        "margin_w":        _pre.margin_width,
        "top_margin":      _pre.top_margin,
        "bottom_margin":   _pre.bottom_margin,
        "binding_w":       _pre.binding_width,
        "n_lines":         _seg.n_lines,
        "text_coverage":   round(_seg.text_coverage, 4),
        "ink_retained":    round(_seg.text_px_kept / max(1, _seg.text_px_input), 3),
        "n_figures":       _seg.n_figures,
        "figure_coverage": round(_seg.figure_coverage, 4),
    })
    print(f"done  ({_seg.n_lines} lines, {_seg.n_figures} figures)")

df = pd.DataFrame(rows)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)
df


---
## Spot-check grid — Preprocess → Segment for a selection of pages

In [ ]:
SPOT_TAGS = [
    "double_simple",
    "single_simple",
    "single_top_double_bottom",
    "double_big_stain",
]

n_pages = len(SPOT_TAGS)
# 3 columns: masked binary | text binary + line polygons | figure binary + bboxes
fig, axes = plt.subplots(n_pages, 3, figsize=(15, 5 * n_pages))
if n_pages == 1:
    axes = axes[np.newaxis, :]

for row_i, tag in enumerate(SPOT_TAGS):
    try:
        path = pick(tag)
    except FileNotFoundError:
        for c in range(3):
            axes[row_i, c].set_visible(False)
        continue

    _pre = preprocess_page(path)
    _seg = segment_kraken(_pre, model_path=MODEL_PATH, device=DEVICE, dilation_px=2)

    # Col 0 — masked binary (Stage 1)
    axes[row_i, 0].imshow(_pre.masked, cmap="gray")
    axes[row_i, 0].set_title(f"Preprocess — {tag}", fontsize=9)

    # Col 1 — Kraken line polygons on BGR
    poly_img = cv2.cvtColor(_pre.bgr, cv2.COLOR_BGR2RGB).copy()
    for boundary in _seg.line_boundaries:
        pts = np.array(boundary, dtype=np.int32).reshape(-1, 1, 2)
        cv2.polylines(poly_img, [pts], isClosed=True, color=(255, 80, 0), thickness=1)
    axes[row_i, 1].imshow(poly_img)
    axes[row_i, 1].set_title(f"Segment — {_seg.n_lines} lines", fontsize=9)

    # Col 2 — figure bounding boxes on BGR
    fig_img = cv2.cvtColor(_pre.bgr, cv2.COLOR_BGR2RGB).copy()
    for x, y, w, h in _seg.figure_bboxes:
        cv2.rectangle(fig_img, (x, y), (x + w, y + h), (255, 60, 0), 2)
    axes[row_i, 2].imshow(fig_img)
    axes[row_i, 2].set_title(f"Figures — {_seg.n_figures} regions", fontsize=9)

    for c in range(3):
        axes[row_i, c].axis("off")

plt.tight_layout()
plt.show()


---
## Running full pipeline and saving results, just to double check

In [ ]:
# Running all the pipeline using the process_image in run_pipeline.py
%reload_ext autoreload
%autoreload 2

from run_pipeline import process_image

SAMPLE_TAGS = [
    "double_simple",
    "double_filigranes",
    "double_big_stain",
    "double_figures",
    "single_simple",
    "single_many_elements",
    "single_top_double_bottom",
    "single_separation_lines",
]

#SAMPLE_TAGS = [
#"double_figures"
#]

for tag in SAMPLE_TAGS:
    try:
        path = pick(tag)
    except FileNotFoundError:
        continue
    print(f"Processing {path.name} ...", end=" ", flush=True)
    
    result = process_image(path, device="mps", min_dimension_px=75, compute_embed = False, min_gutter_fraction=0.70)

In [ ]:
%reload_ext autoreload
%autoreload 2

from run_pipeline import process_image
from pathlib import Path
path = Path("/Users/luissalamanca/Dropbox/My_stuff/05_SDSCresearch/10_SideProjects/00_MedievalCambridge/line_counting/data/all_images/MS-GG-00001-00001-000-00447.jpg")

result = process_image(path, device="mps", compute_embed = True, dark_threshold=160, min_dimension_px=70)

In [ ]:
result["pre"].binding_width

In [ ]:
from pipeline.plots import plot_postprocess
fig = plot_postprocess(result["post"],result["pre"], result["seg"])

In [ ]:
%reload_ext autoreload
%autoreload 2

from pipeline.stages import preprocess_page
from pipeline.plots  import plot_preprocess
from pathlib import Path

path = Path("/Users/luissalamanca/Dropbox/My_stuff/05_SDSCresearch/10_SideProjects/00_MedievalCambridge/line_counting/data/all_images/MS-GG-00001-00001-000-00039.jpg")

pre = preprocess_page(path)


In [ ]:
fig = plot_preprocess(pre)
fig.suptitle(f"Preprocess", fontsize=13, fontweight="bold")
